In [1]:
from dotenv import load_dotenv

load_dotenv()

True

### rag_with_chroma
1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
 - 토큰수 초과로 답변을 생서하지 못할 수 있고
  - 문서가 길면 (input이 길면) 답변 생성이 오래 걸림
3. 임베딩 -> 벡터 데이터베이스에 저장
4. 질문이 있을 때, 벡터 데이터 베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달

In [2]:
#1. 문서의 내용을 읽는다.(docx2txt)
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200
)

loader=Docx2txtLoader('./inflearn-rag-notebook/tax_with_markdown.docx')
document_list=loader.load_and_split(text_splitter=text_splitter)

/tmp/ipykernel_192341/2132813693.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import Docx2txtLoader


In [3]:
len(document_list)

225

In [4]:
from langchain_openai import ChatOpenAI
llm=ChatOpenAI(model='gpt-4o')

In [5]:
from langchain_openai import OpenAIEmbeddings
embedding=OpenAIEmbeddings(model='text-embedding-3-large', dimensions=1024)

In [6]:
import os
pinecone_api_key=os.environ.get("PINECONE_API_KEY")

import time
from langchain_pinecone import PineconeVectorStore
from pinecone import Pinecone, ServerlessSpec
pc= Pinecone(api_key=pinecone_api_key)

# 고정 id를 주면 셀을 다시 실행해도 중복 추가되지 않고 덮어씀(upsert)
ids=[f"tax-{i}" for i in range(len(document_list))]
database=PineconeVectorStore.from_documents(documents=document_list, embedding=embedding, ids=ids, index_name='tax-index')

/home/yutae/work/Agent/inflearn/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

dictionary=["사람을 나타내는 표현 -> 거주자"]
prompt=ChatPromptTemplate.from_template(f"""
            사용자의 질문을 보고, 우리의 사전을 참고해서 사용자의 질문을 변경해 주세요.
            만약 변경할 필요가 없다고 판단된다면, 사용자의 질문을 변경하지 않아도 됩니다.
            사전: {dictionary}
            
            질문: {{question}}""")
dictionary_chain= prompt|llm|StrOutputParser()
                                        

In [8]:
query='연봉 5천만원인 직장인의 소득세는 얼마인가요?'
new_question=dictionary_chain.invoke({"question":query})
new_question

'연봉 5천만원인 거주자의 소득세는 얼마인가요?'

In [9]:
#langsmith hub 이용
from langsmith import Client

client = Client()
prompt = client.pull_prompt("rlm/rag-prompt", dangerously_pull_public_prompt=True)

from langchain_classic.chains import RetrievalQA

qa_chain=RetrievalQA.from_chain_type(
    llm,
    retriever=database.as_retriever(),
    chain_type_kwargs={'prompt':prompt}
)

In [10]:
tax_chain={'query':dictionary_chain}| qa_chain

In [11]:
ai_response=tax_chain.invoke({"question": query})
ai_response

{'query': '연봉 5천만원인 거주자의 소득세는 얼마인가요?',
 'result': '연봉 5천만 원인 거주자의 소득세는 기본적으로 624만 원입니다. 이는 "5,000만원 이하"에 해당하는 세율 구간에 따라 계산된 것입니다. 상세 비용은 추가적인 공제 항목에 따라 달라질 수 있습니다.'}